In [ ]:
# nanoGPT from scratch on Kaggle
# 第 1 个 cell：环境检查、目标说明、导入依赖、固定随机种子
#
# 这份 notebook 的目标不是直接调用官方 nanoGPT 的 train.py，
# 而是在 Kaggle 里一步一步“自己写出一个 nanoGPT”。
# 官方源码已经放在仓库的 nanoGPT-reference/ 目录中，后续每一步都会对照它来实现。
#
# 当前第 1 格只做准备工作：
# 1. 确认 PyTorch 和 GPU 状态
# 2. 处理 Kaggle 上 P100 + 新版 PyTorch 不兼容的问题
# 3. 导入后续会用到的基础库
# 4. 固定随机种子，方便复现实验
# 5. 设置一个 device 变量，后续张量和模型都放到这个设备上

import math
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


def choose_device():
    """选择当前 Kaggle 环境里真正能安全使用的设备。"""
    if not torch.cuda.is_available():
        print("GPU not found, using CPU. 也能跑小实验，但会慢一些。")
        return "cpu"

    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    print("GPU:", name)
    print("CUDA capability:", f"sm_{props.major}{props.minor}")

    # Kaggle 有时会分配 Tesla P100。P100 是 sm_60。
    # 你遇到的 PyTorch 2.10.0+cu128 构建只支持 sm_70 及以上。
    # 这种情况下 torch.cuda.is_available() 仍然可能是 True，
    # 但真正执行 CUDA kernel 时会失败，所以这里主动退回 CPU。
    if props.major < 7:
        print("\n注意：当前 GPU 是 P100/sm_60，但这个 PyTorch 构建不支持 sm_60。")
        print("本 notebook 将临时使用 CPU。想用 GPU，请在 Kaggle Accelerator 里优先选择 T4。")
        print("如果 Kaggle 只能给 P100，也可以改装兼容旧架构的 PyTorch，但初学阶段先不折腾环境。\n")
        return "cpu"

    return "cuda"


device = choose_device()


# 固定随机种子。
# 训练神经网络时，初始化权重、打乱数据、采样 token 都会用到随机数。
# 固定 seed 后，每次从头运行 notebook，结果更容易复现和对比。
seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device == "cuda":
    torch.cuda.manual_seed(seed)


# 允许 TensorFloat-32。它是 NVIDIA GPU 上的一种加速格式。
# 对我们这种学习实验来说，可以让矩阵乘法更快。
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


print("Using device:", device)
print("Seed:", seed)


# 后续我们会按这个顺序继续写：
# Cell 2：按官方路线准备 OpenWebText，并用 GPT-2 BPE 变成 token
# Cell 3：写 get_batch，构造 x/y 训练样本
# Cell 4：先实现 BigramLanguageModel，理解最小语言模型
# Cell 5：实现单头 causal self-attention
# Cell 6：实现多头注意力
# Cell 7：实现 MLP 和 Block
# Cell 8：组装 GPTConfig 和 GPT
# Cell 9：写训练循环
# Cell 10：写 generate 生成文本


In [ ]:
# 第 2 个 cell：官方 OpenWebText + GPT-2 BPE 数据准备
#
# 这一格对应 nanoGPT 官方源码：
#   nanoGPT-reference/data/openwebtext/prepare.py
#
# 现在我们不再使用本地小文本文件，也不再使用 Tiny Shakespeare。
# 如果目标是尽量复现 nanoGPT 官方 GPT-2 结果，数据路线必须是：
#
#   OpenWebText -> GPT-2 BPE tokenizer -> train.bin / val.bin
#
# OpenWebText 是 OpenAI 私有 WebText 的开源近似复刻版。
# GPT-2 原论文训练用的 WebText 没有公开，所以 nanoGPT 官方用 OpenWebText 来复现。
#
# 重要提醒：
# 官方 OpenWebText 数据准备很大。nanoGPT 官方注释里写到：
#   Hugging Face cache 约 54GB
#   train.bin 约 17GB
#   val.bin 约 8.5MB
#   train 约 9B tokens
#
# 所以这不是玩具数据。Kaggle 如果磁盘或时间不够，可能会失败。
# 但从“和官方方案一致”的角度看，这就是正确路线。

import os
import subprocess
import sys
from pathlib import Path

import numpy as np


def ensure_package(import_name, pip_name=None):
    """缺少依赖时自动安装，方便在 Kaggle 新环境中直接运行。"""
    try:
        return __import__(import_name)
    except ModuleNotFoundError:
        package_name = pip_name or import_name
        print(f"当前环境没有 {package_name}，正在安装。")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return __import__(import_name)


tiktoken = ensure_package("tiktoken")
ensure_package("datasets")
ensure_package("tqdm")
from datasets import load_dataset
from tqdm import tqdm


# 和官方 openwebtext/prepare.py 保持一致：使用 GPT-2 BPE。
# BPE 可以理解成“常见文本片段优先合并”的分词方式。
# 它不是一个字符一个 token，而是把常见单词、空格加单词、子词片段变成 token。
#
# GPT-2 BPE 的普通词表大小是 50257。
# eot_token 是 end of text，编号通常是 50256，用来告诉模型“一篇文章到这里结束了”。

enc = tiktoken.get_encoding("gpt2")
gpt2_vocab_size = enc.n_vocab
vocab_size = 50304  # nanoGPT 从零训练时默认把 50257 padding 到 64 的倍数，矩阵计算更整齐。

print("Tokenizer:", enc.name)
print("GPT-2 原始词表大小:", gpt2_vocab_size)
print("GPT-2 eot_token:", enc.eot_token)
print("nanoGPT 从零训练默认 vocab_size:", vocab_size)


# 看一个很小的 BPE 例子。
# token id 只是词表里的编号，不代表大小或重要程度。

example_text = "The quick brown fox jumps over the lazy dog."
example_ids = enc.encode_ordinary(example_text)

print("\nBPE 编码/解码小例子：")
print("原始文本:", example_text)
print("编码后 token id:", example_ids)
print("解码回去:", enc.decode(example_ids))
print("每个 token id 对应的文本片段：")
for token_id in example_ids:
    print(f"{token_id:>6} -> {enc.decode([token_id])!r}")


# 输出目录放在 Kaggle 的工作目录。
# 后面训练时会从这里读取：
#   /kaggle/working/data/openwebtext/train.bin
#   /kaggle/working/data/openwebtext/val.bin
#
# 如果在本地运行，则会写到当前项目的 data/openwebtext。

if Path("/kaggle/working").exists():
    data_dir = Path("/kaggle/working/data/openwebtext")
else:
    data_dir = Path("data/openwebtext")
data_dir.mkdir(parents=True, exist_ok=True)

print("\n数据输出目录:", data_dir)


# num_proc 是并行进程数。
# 官方源码默认 8。这里最多用 8，同时照顾 Kaggle CPU 数量。

num_proc = min(8, max(1, (os.cpu_count() or 2) // 2))
num_proc_load_dataset = num_proc

print("并行进程数 num_proc:", num_proc)


# 这一步会下载并加载完整 OpenWebText。
# 官方数据集只有 train split，所以官方用 train_test_split 人工切出 val。
#
# test_size=0.0005 的意思是：
#   从训练数据里拿出 0.05% 当验证集。
# seed=2357 和 shuffle=True 也照抄官方，保证切分方式一致。

print("\n开始加载 OpenWebText。第一次运行会下载大量数据，请耐心等待。")
dataset = load_dataset("openwebtext", num_proc=num_proc_load_dataset)

split_dataset = dataset["train"].train_test_split(
    test_size=0.0005,
    seed=2357,
    shuffle=True,
)
split_dataset["val"] = split_dataset.pop("test")

print("\n切分后的数据集：")
print(split_dataset)


def process(example):
    """把一篇 OpenWebText 文档转换成 GPT-2 BPE token id。"""
    ids = enc.encode_ordinary(example["text"])

    # 官方 openwebtext/prepare.py 会在每篇文档末尾追加 eot_token。
    # 这相当于给模型一个边界标记：
    #   前一篇文章结束了，下一篇文章不要被当成同一段连续文字。
    ids.append(enc.eot_token)

    return {"ids": ids, "len": len(ids)}


print("\n开始 GPT-2 BPE 分词。")
tokenized = split_dataset.map(
    process,
    remove_columns=["text"],
    desc="tokenizing the splits",
    num_proc=num_proc,
)


# 把 token 写成 train.bin / val.bin。
#
# 为什么要写成 .bin？
# 因为 OpenWebText 太大了，不能每次训练都重新分词。
# 写成连续的 uint16 二进制文件后，训练时可以用 np.memmap 像翻书一样读取局部片段。
#
# 为什么是 uint16？
# GPT-2 token id 最大是 50256，小于 65535，刚好可以用 16 位无符号整数保存。
# 这样比 int64 更省空间。

bin_paths = {}
for split, dset in tokenized.items():
    arr_len = np.sum(dset["len"], dtype=np.uint64)
    filename = data_dir / f"{split}.bin"
    dtype = np.uint16
    arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))

    total_batches = 1024
    idx = 0

    print(f"\n开始写入 {filename}")
    print(f"{split} token 总数:", f"{int(arr_len):,}")

    for batch_idx in tqdm(range(total_batches), desc=f"writing {filename.name}"):
        batch = dset.shard(
            num_shards=total_batches,
            index=batch_idx,
            contiguous=True,
        ).with_format("numpy")
        arr_batch = np.concatenate(batch["ids"])
        arr[idx : idx + len(arr_batch)] = arr_batch
        idx += len(arr_batch)

    arr.flush()
    bin_paths[split] = filename


# 后面训练时不需要把完整 train.bin 读进内存。
# np.memmap 会建立“文件视图”，用到哪一段再读哪一段。

train_bin_path = bin_paths["train"]
val_bin_path = bin_paths["val"]

train_data = np.memmap(train_bin_path, dtype=np.uint16, mode="r")
val_data = np.memmap(val_bin_path, dtype=np.uint16, mode="r")

print("\nOpenWebText 数据准备完成。")
print("train.bin:", train_bin_path, "tokens:", f"{len(train_data):,}")
print("val.bin:", val_bin_path, "tokens:", f"{len(val_data):,}")
print("训练集前 40 个 token id:")
print(train_data[:40].tolist())
print("解码前 40 个 token:")
print(enc.decode(train_data[:40].tolist()))


def encode(s):
    """把字符串变成 GPT-2 BPE token id 列表。"""
    return enc.encode_ordinary(s)


def decode(ids):
    """把 GPT-2 BPE token id 列表还原成字符串。"""
    return enc.decode(ids)


# 这一格产生的关键变量：
#   enc / encode / decode       GPT-2 BPE 分词器和辅助函数
#   vocab_size                  nanoGPT 从零训练用的词表大小，50304
#   data_dir                    OpenWebText 二进制数据目录
#   train_bin_path / val_bin_path
#   train_data / val_data       np.memmap 形式的 token 序列
#
# 下一格 Cell 3 会讲：
#   如何从 train_data 中随机取一批连续 token，
#   构造 x 和 y，让模型学习“看到前文，预测下一个 token”。
